# Local conformational variation with RFdiffusion3

This notebook adapts the Binder/Refold AGP experiment into a reusable Colab workflow. It generates local conformational alternatives and collects accepted structures with a fixed, jointly evaluated glycosylation set while keeping the surrounding scaffold fixed.

## Quick start

1. Run the notebook from top to bottom. The setup step installs the required packages and RFD3 checkpoint.
2. Select **Sequence** or **PDB/mmCIF structure**. For a structure, leave the optional path blank and run the upload step to choose one `.pdb`, `.cif`, or `.mmcif` file.
3. Run **Inspect input chains and residue numbering**. For a structure, it shows a 3D preview plus each chain's observed residue IDs and sequence before you choose a target.
4. Set the target/window and accepted-count controls in the next form, then run the remaining cells. The configured glycans are attached together in one ReGlyco build, so glycan–glycan clashes reject the whole candidate. Sequence substitutions can be entered there; structural inputs must already contain a chemically modeled mutant.
5. Inspect the aligned ensemble in Mol*, then use the buttons at the end to download individual files or a ZIP bundle.

Implementation code is collapsed by default for a cleaner Colab interface; use Colab's **Show code** control whenever you want to inspect it. This is a visual convenience, not code protection.

The default example is the 181-residue AGP sequence used for the Asn36 experiment (`N36`, window `A30–A42`). Replace it with your sequence or a structure. For sequence-mode mutations, use one-letter notation such as `N36A` or `N36A,S38T`; the reference residue is checked before RF3 folding. For a structure-mode mutation, upload a chemically modeled/relaxed mutant structure—renaming a residue in a coordinate file is not a valid side-chain model. Include glycans or other components in the uploaded structure if they should be retained as context.

## Methodology

1. **Prepare a starting structure.** A sequence (with any requested substitutions) is folded once with RF3. A PDB/mmCIF input is used with its chain and residue numbering.
2. **Define the local region.** Choose a target residue and a residue window. Atoms outside that window are selected as fixed scaffold anchors.
3. **Generate local alternatives with partial diffusion.** RFD3 perturbs the selected region around the input structure while `select_fixed_atoms` holds the outside scaffold in place and `select_unfixed_sequence=False` keeps sequence identity fixed. `PARTIAL_T` is the noise scale in Å—not elapsed time or a count of steps. `EXPOSURE_CONDITION` maps to exposed-surface conditioning; `LOOP_BIAS` maps to `is_non_loopy` (`False` requests more loops, `True` fewer). These are model guidance, not guarantees. The RFD3 input guide recommends starting conservatively; this notebook defaults to 12 Å.
4. **Assess candidates.** The notebook samples independent structures, annotates secondary structure with P-SEA, and checks for large adjacent Cα gaps and any clash count reported in RFD3 metadata. `REQUIRE_TARGET_NONHELICAL` is a post-generation acceptance filter requiring a coil call at the target; it does not directly force the model to produce one. This is a filter, not a physics-based energy ranking; all generated candidates remain available for inspection.
5. **Align for comparison.** By default, each model is rigidly Kabsch-aligned to the original using common protein Cα atoms outside the selected window. This removes overall pose differences without deforming the structures, making the local change easier to see.

> These are model-generated structural hypotheses, not a molecular-dynamics trajectory, a measured unfolding pathway, or equilibrium populations. Do not interpret the fraction of accepted samples as a thermodynamic probability. Inspect geometry and validate candidates independently before drawing mechanistic conclusions.

For definitions of `partial_t`, fixed-atom selections, exposure conditioning, and loop guidance, see the [official RFD3 input specification](https://github.com/RosettaCommons/foundry/blob/production/models/rfd3/docs/input.md).

In [ ]:
#@title Install dependencies and the RFD3 checkpoint { display-mode: "form" }
import importlib.util
import os
import shutil
import subprocess
import sys
from pathlib import Path

os.environ.setdefault('CCD_MIRROR_PATH', '')
os.environ.setdefault('PDB_MIRROR_PATH', '')
# Some Colab CUDA images expose a cuEquivariance binary incompatible with the selected GPU.
# RF3's vanilla PyTorch attention is slower but portable and avoids the CUDA symbol failure.
os.environ['DISABLE_CUEQUIVARIANCE'] = '1'
print('cuEquivariance disabled for Colab compatibility; RF3 will use vanilla PyTorch attention.')
WORKDIR = Path('/content/local_conformational_ensemble')
WORKDIR.mkdir(parents=True, exist_ok=True)

# The helper is kept in GlycoShape-Resources/colab and fetched as a pinned
# source file when this notebook is opened directly in Colab.  It fixes the
# provider settings internally; no provider-level control is exposed here.
HELPER_URL = 'https://raw.githubusercontent.com/Ojas-Singh/GlycoShape-Resources/main/colab/reglyco_local.py'
RELEASE_METADATA_URL = 'https://raw.githubusercontent.com/Ojas-Singh/GlycoShape-Resources/main/colab/reglyco_release.json'
HELPER_PATH = WORKDIR / 'reglyco_local.py'
RELEASE_METADATA_PATH = WORKDIR / 'reglyco_release.json'
if not HELPER_PATH.exists() or not RELEASE_METADATA_PATH.exists():
    import urllib.request
    try:
        with urllib.request.urlopen(HELPER_URL, timeout=60) as response:
            HELPER_PATH.write_bytes(response.read())
        with urllib.request.urlopen(RELEASE_METADATA_URL, timeout=60) as response:
            RELEASE_METADATA_PATH.write_bytes(response.read())
    except Exception as exc:
        raise RuntimeError(f'Could not download the local ReGlyco adapter/manifest from {HELPER_URL}: {exc}') from exc
if str(WORKDIR) not in sys.path:
    sys.path.insert(0, str(WORKDIR))
from reglyco_local import (
    ReGlycoCommandError,
    clash_status,
    is_clash_free,
    output_structure,
    provenance,
    read_json,
    run_reglyco,
)
print('Local ReGlyco adapter:', HELPER_PATH)

def run_checked(command):
    print('$', ' '.join(str(item) for item in command))
    subprocess.check_call(command)

if importlib.util.find_spec('rfd3') is None:
    print('Installing rc-foundry (AtomWorks + RFD3 runtime)...')
    run_checked([sys.executable, '-m', 'pip', 'install', '-q', 'rc-foundry[all]'])
else:
    print('rc-foundry/RFD3 Python package is already available.')

if importlib.util.find_spec('biotite') is None or importlib.util.find_spec('pandas') is None:
    run_checked([sys.executable, '-m', 'pip', 'install', '-q', 'biotite', 'pandas'])

foundry = shutil.which('foundry')
if foundry is None:
    raise RuntimeError('The foundry command was not found after installation. Restart the Colab runtime and run this cell again.')

rfd3_marker = Path('/content/.glycoshape_rfd3_checkpoint_ready')
if not rfd3_marker.exists():
    run_checked([foundry, 'install', 'rfd3'])
    rfd3_marker.write_text('ok\n')
else:
    print('RFD3 checkpoint marker found; using the cached checkpoint.')

try:
    import torch
    print('PyTorch:', torch.__version__)
    print('CUDA:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'not available')
except Exception as exc:
    print('Could not query CUDA:', exc)

In [ ]:
#@title Step 1: Choose a sequence or structure { display-mode: "form" }
INPUT_MODE = 'Sequence' #@param ["Sequence", "PDB/mmCIF structure"]
UPLOADED_STRUCTURE_PATH = None  # Reset the upload whenever these settings are run.
PROTEIN_SEQUENCE = 'PLCANLVPVPITNATLDRITGKWFYIASAFRNEEYNKSVQEIQATFFYFTPNKTEDTIFLREYQTRQDQCIYNTTYLNVQRENGTISRYVGGQEHFAHLLILRDTKTYMLAFDVNDEKNWGLSVYADKPETTKEQLGEFYEALDCLRIPKSDVVYTDWKKDKCEPLEKQHEKERKQEEGES' #@param {type:"string"}
STRUCTURE_PATH = '' #@param {type:"string"}
# Optional existing file path; normally leave blank and use the upload step below.
print('Input mode:', INPUT_MODE)
if INPUT_MODE == 'Sequence':
    print('Enter or replace the protein sequence above; residue and mutation settings come after sequence inspection.')
else:
    print('Run the next step to upload a PDB/mmCIF; chain and residue choices will be shown afterward.')

In [ ]:
#@title Step 2: Upload PDB or mmCIF (only for structure input)
from pathlib import Path

UPLOADED_STRUCTURE_PATH = None
if INPUT_MODE == 'Sequence':
    print('Sequence mode selected — no file upload is needed.')
elif STRUCTURE_PATH.strip():
    UPLOADED_STRUCTURE_PATH = Path(STRUCTURE_PATH).expanduser()
    if not UPLOADED_STRUCTURE_PATH.is_file():
        raise FileNotFoundError(f'Structure file not found: {UPLOADED_STRUCTURE_PATH}')
    if UPLOADED_STRUCTURE_PATH.suffix.lower() not in {'.pdb', '.ent', '.cif', '.mmcif'}:
        raise ValueError('Structure path must end in .pdb, .ent, .cif, or .mmcif.')
    print('Using structure already in the runtime:', UPLOADED_STRUCTURE_PATH)
else:
    from google.colab import files
    print('Select exactly one structure file: .pdb, .cif, or .mmcif.')
    uploaded_files = files.upload()
    if not uploaded_files:
        raise ValueError('No file selected. Rerun this upload step and choose one PDB/mmCIF file.')
    if len(uploaded_files) != 1:
        raise ValueError(f'Please upload exactly one structure file; received {len(uploaded_files)} files.')
    uploaded_name, uploaded_bytes = next(iter(uploaded_files.items()))
    uploaded_name = Path(uploaded_name).name
    if Path(uploaded_name).suffix.lower() not in {'.pdb', '.ent', '.cif', '.mmcif'}:
        raise ValueError(f'{uploaded_name!r} is not a supported structure file. Choose .pdb, .cif, or .mmcif.')
    UPLOADED_STRUCTURE_PATH = WORKDIR / uploaded_name
    UPLOADED_STRUCTURE_PATH.write_bytes(uploaded_bytes)
    print(f'Uploaded and ready: {UPLOADED_STRUCTURE_PATH.name} ({UPLOADED_STRUCTURE_PATH.stat().st_size:,} bytes)')

In [ ]:
#@title Step 3: Load analysis and visualization helpers
import base64
import json
import math
import re
import uuid
from dataclasses import dataclass

import biotite.structure as struc
import numpy as np
import pandas as pd
from biotite.structure.io.pdb import PDBFile
from biotite.structure.sse import annotate_sse
from IPython.display import HTML, display

AA1_TO_AA3 = {
    'A': 'ALA', 'R': 'ARG', 'N': 'ASN', 'D': 'ASP', 'C': 'CYS',
    'Q': 'GLN', 'E': 'GLU', 'G': 'GLY', 'H': 'HIS', 'I': 'ILE',
    'L': 'LEU', 'K': 'LYS', 'M': 'MET', 'F': 'PHE', 'P': 'PRO',
    'S': 'SER', 'T': 'THR', 'W': 'TRP', 'Y': 'TYR', 'V': 'VAL',
}
AA3_TO_AA1 = {three: one for one, three in AA1_TO_AA3.items()}

@dataclass(frozen=True)
class ResidueRecord:
    chain_id: str
    res_id: int
    res_name: str

def validate_sequence(sequence):
    lines = [line.strip() for line in str(sequence).splitlines() if not line.strip().startswith('>')]
    cleaned = ''.join(lines).replace(' ', '').replace('\t', '').upper()
    invalid = sorted(set(cleaned) - set(AA1_TO_AA3))
    if not cleaned:
        raise ValueError('The sequence is empty.')
    if invalid:
        raise ValueError(f'Invalid amino-acid characters: {invalid}')
    return cleaned

def parse_mutations(text):
    if not str(text).strip():
        return []
    mutations = []
    for token in re.split(r'[,;\s]+', str(text).strip()):
        match = re.fullmatch(r'([A-Z])(\d+)([A-Z])', token.upper())
        if match is None:
            raise ValueError(f'Could not parse mutation {token!r}; use one-letter notation such as N36A.')
        reference, position, mutant = match.groups()
        mutations.append((reference, int(position), mutant))
    return mutations

def apply_mutations(sequence, mutation_text):
    result = list(validate_sequence(sequence))
    applied = []
    for reference, position, mutant in parse_mutations(mutation_text):
        if position < 1 or position > len(result):
            raise ValueError(f'Mutation position {position} is outside the sequence (length {len(result)}).')
        observed = result[position - 1]
        if observed != reference:
            raise ValueError(f'Mutation {reference}{position}{mutant} does not match the sequence; observed {observed}{position}.')
        result[position - 1] = mutant
        applied.append(f'{reference}{position}{mutant}')
    return ''.join(result), applied

def load_structure_file(path):
    path = Path(path)
    suffix = path.suffix.lower()
    if suffix in {'.cif', '.mmcif'}:
        try:
            from biotite.structure.io.pdbx import CIFFile, get_structure
        except ImportError:
            from biotite.structure.io.pdbx import PDBxFile as CIFFile, get_structure
        parsed = CIFFile.read(path)
        atom_array = get_structure(parsed, model=1)
    else:
        atom_array = PDBFile.read(path).get_structure(model=1)
    if isinstance(atom_array, struc.AtomArrayStack):
        atom_array = atom_array[0]
    return atom_array

def as_atom_array(value):
    if isinstance(value, struc.AtomArrayStack):
        return value[0]
    return value

def pdb_compatible_array(value):
    """Return a copy whose B-factors fit the fixed-width legacy PDB field."""
    array = as_atom_array(value).copy()
    if hasattr(array, 'b_factor'):
        b_factor = np.asarray(array.b_factor, dtype=float)
        valid = np.isfinite(b_factor) & (b_factor >= 0.0) & (b_factor <= 999.99)
        if not np.all(valid):
            safe_b_factor = b_factor.copy()
            safe_b_factor[~valid] = 0.0
            array.b_factor = safe_b_factor
    return array

def write_single_pdb(atom_array, path):
    pdb = PDBFile()
    pdb.set_structure(pdb_compatible_array(atom_array))
    pdb.write(path)

def build_molstar_structure_preview_html(pdb_path, height=520):
    pdb_text = Path(pdb_path).read_text()
    pdb_base64 = base64.b64encode(pdb_text.encode('utf-8')).decode('ascii')
    viewer_id = f'uploaded-structure-preview-{uuid.uuid4().hex}'
    loading_id = f'{viewer_id}-loading'
    html = '''
<style>
.uploaded-structure-preview { width:100%; position:relative; border:1px solid #dbe3ec; border-radius:12px; overflow:hidden; }
.uploaded-structure-loading { position:absolute; inset:0; z-index:5; display:flex; align-items:center; justify-content:center; background:#fff; color:#64748b; font:12px system-ui,sans-serif; }
</style>
<link rel='stylesheet' href='https://cdn.jsdelivr.net/npm/molstar@3/build/viewer/molstar.css'>
<script src='https://cdn.jsdelivr.net/npm/molstar@3/build/viewer/molstar.js'></script>
<div id='__VIEWER_ID__' class='uploaded-structure-preview' style='height:__HEIGHT__px'><div id='__LOADING_ID__' class='uploaded-structure-loading'>LOADING MOL*</div></div>
<script>
(() => {
  const pdbText = new TextDecoder().decode(Uint8Array.from(atob('__PDB_BASE64__'), c => c.charCodeAt(0)));
  const viewerId = '__VIEWER_ID__';
  const loadingId = '__LOADING_ID__';
  const byId = id => document.getElementById(id);
  async function init() {
    try {
      const viewer = await molstar.Viewer.create(viewerId, { layoutIsExpanded:false, layoutShowControls:true, layoutShowRemoteState:false, layoutShowSequence:true, layoutShowLog:false });
      const data = await viewer.plugin.builders.data.rawData({ data:pdbText, label:'Uploaded input structure' });
      const trajectory = await viewer.plugin.builders.structure.parseTrajectory(data, 'pdb');
      await viewer.plugin.builders.structure.hierarchy.applyPreset(trajectory, 'all-models', { useDefaultIfSingleModel:true, showUnitcell:false, representationPreset:'auto' });
      const loading = byId(loadingId);
      if (loading) loading.style.display = 'none';
    } catch (error) {
      console.error('Mol* input preview failed:', error);
      const loading = byId(loadingId);
      if (loading) loading.textContent = 'Mol* preview failed to load; use the chain/residue listing below.';
    }
  }
  function waitForMolstar(tries=0) {
    if (typeof molstar !== 'undefined' && molstar.Viewer) { init(); return; }
    if (tries < 60) { setTimeout(() => waitForMolstar(tries + 1), 200); return; }
    const loading = byId(loadingId);
    if (loading) loading.textContent = 'Mol* JavaScript did not load; use the chain/residue listing below.';
  }
  waitForMolstar();
})();
</script>
'''
    return (html.replace('__PDB_BASE64__', pdb_base64)
                .replace('__VIEWER_ID__', viewer_id)
                .replace('__LOADING_ID__', loading_id)
                .replace('__HEIGHT__', str(int(height))))

def residue_records(atom_array, protein_only=False):
    array = as_atom_array(atom_array)
    if protein_only:
        array = protein_array(array)
    starts = struc.get_residue_starts(array)
    records = []
    seen = set()
    for start in starts:
        key = (str(array.chain_id[start]), int(array.res_id[start]))
        if key in seen:
            continue
        seen.add(key)
        records.append(ResidueRecord(key[0], key[1], str(array.res_name[start])))
    return records

def protein_array(atom_array):
    array = as_atom_array(atom_array)
    array = array[struc.filter_amino_acids(array)]
    # RFD3 separates HETATM/non-polymer residues (e.g. 3KQ0's PCA A1) from polymer chains.
    # Exclude them from chain/residue selectors so fixed ranges match RFD3's parsed polymer IDs.
    if 'hetero' in array.get_annotation_categories():
        array = array[~array.hetero]
    return array

def residue_index(records, chain_id, res_id):
    for index, record in enumerate(records):
        if record.chain_id == str(chain_id) and record.res_id == int(res_id):
            return index
    raise ValueError(f'Residue {chain_id}{res_id} was not found.')

def residue_ids_to_ranges(residue_ids):
    values = sorted(set(int(value) for value in residue_ids))
    if not values:
        return []
    ranges = []
    start = end = values[0]
    for value in values[1:]:
        if value == end + 1:
            end = value
        else:
            ranges.append((start, end))
            start = end = value
    ranges.append((start, end))
    return ranges

def make_fixed_selection(records, chain_id, window_start, window_end):
    grouped = {}
    for record in records:
        inside_window = (record.chain_id == chain_id and window_start <= record.res_id <= window_end)
        if not inside_window:
            grouped.setdefault(record.chain_id, []).append(record.res_id)
    parts = []
    for chain in sorted(grouped):
        for start, end in residue_ids_to_ranges(grouped[chain]):
            parts.append(f'{chain}{start}' if start == end else f'{chain}{start}-{end}')
    if not parts:
        raise ValueError('The selected window covers the entire input; leave a fixed scaffold outside it.')
    return ','.join(parts)

def detect_window(atom_array, chain_id, target_residue, padding, fallback_half_window):
    array = protein_array(atom_array)
    records = residue_records(array)
    target = residue_index(records, chain_id, target_residue)
    sse = annotate_sse(array)
    if sse[target] == 'a':
        left = target
        while left > 0 and records[left - 1].chain_id == chain_id and sse[left - 1] == 'a':
            left -= 1
        right = target
        while right + 1 < len(records) and records[right + 1].chain_id == chain_id and sse[right + 1] == 'a':
            right += 1
        chain_positions = [i for i, record in enumerate(records) if record.chain_id == chain_id]
        left = max(left - int(padding), chain_positions[0])
        right = min(right + int(padding), chain_positions[-1])
        return records[left].res_id, records[right].res_id, 'detected helix'
    chain_positions = [i for i, record in enumerate(records) if record.chain_id == chain_id]
    chain_target = chain_positions.index(target)
    left = max(chain_target - int(fallback_half_window), 0)
    right = min(chain_target + int(fallback_half_window), len(chain_positions) - 1)
    return records[chain_positions[left]].res_id, records[chain_positions[right]].res_id, 'fallback window'

def disable_cuequivariance():
    """Force portable attention, including after a previous failed import in this runtime."""
    os.environ['DISABLE_CUEQUIVARIANCE'] = '1'
    foundry_module = sys.modules.get('foundry')
    if foundry_module is not None:
        foundry_module.SHOULD_USE_CUEQUIVARIANCE = False
    attention_module = sys.modules.get('rf3.model.layers.attention')
    if attention_module is not None:
        attention_module.SHOULD_USE_CUEQUIVARIANCE = False

def ensure_rf3_available():
    if importlib.util.find_spec('rf3') is None:
        run_checked([sys.executable, '-m', 'pip', 'install', '-q', 'rc-foundry[all]'])
    foundry = shutil.which('foundry')
    if foundry is None:
        raise RuntimeError('The foundry command is unavailable; rerun the setup cell.')
    marker = Path('/content/.glycoshape_rf3_checkpoint_ready')
    if not marker.exists():
        run_checked([foundry, 'install', 'rf3'])
        marker.write_text('ok\n')

def fold_sequence(sequence, example_id='sequence_input'):
    disable_cuequivariance()
    ensure_rf3_available()
    from atomworks.io.tools.inference import components_to_atom_array
    from rf3.inference_engines.rf3 import RF3InferenceEngine
    from rf3.utils.inference import InferenceInput
    components = [{'seq': sequence, 'chain_id': 'A'}]
    placeholder = components_to_atom_array(components)
    inference_input = InferenceInput.from_atom_array(placeholder, example_id=example_id)
    engine = RF3InferenceEngine(ckpt_path='rf3', verbose=False)
    result = engine.run(inputs=inference_input)[example_id][0]
    return as_atom_array(result.atom_array)

def ca_coordinates(atom_array):
    array = protein_array(atom_array)
    coordinates = {}
    for index in np.where(array.atom_name == 'CA')[0]:
        key = (str(array.chain_id[index]), int(array.res_id[index]))
        coordinates[key] = np.asarray(array.coord[index], dtype=float)
    return coordinates

def bend_angle(array_a, array_b, array_c):
    first = np.asarray(array_a) - np.asarray(array_b)
    second = np.asarray(array_c) - np.asarray(array_b)
    denominator = np.linalg.norm(first) * np.linalg.norm(second)
    if denominator == 0:
        return float('nan')
    cosine = np.clip(np.dot(first, second) / denominator, -1.0, 1.0)
    return float(np.degrees(np.arccos(cosine)))

def count_ca_chainbreaks(atom_array, threshold=4.5):
    records = residue_records(protein_array(atom_array))
    ca = ca_coordinates(atom_array)
    count = 0
    for previous, current in zip(records, records[1:]):
        if previous.chain_id != current.chain_id or current.res_id != previous.res_id + 1:
            continue
        first = ca.get((previous.chain_id, previous.res_id))
        second = ca.get((current.chain_id, current.res_id))
        if first is not None and second is not None and np.linalg.norm(second - first) > threshold:
            count += 1
    return count

def find_numeric_metric(value, names):
    names = {name.lower() for name in names}
    if isinstance(value, dict):
        for key, item in value.items():
            if str(key).lower() in names and isinstance(item, (int, float, np.integer, np.floating)) and not isinstance(item, bool):
                return float(item)
            found = find_numeric_metric(item, names)
            if found is not None:
                return found
    elif isinstance(value, (list, tuple)):
        for item in value:
            found = find_numeric_metric(item, names)
            if found is not None:
                return found
    return None

def analyze_structure(atom_array, original_ca, metadata, chain_id, target_residue, window_start, window_end):
    array = protein_array(atom_array)
    records = residue_records(array)
    sse = annotate_sse(array)
    code_by_residue = {(record.chain_id, record.res_id): (sse[i] or '?') for i, record in enumerate(records)}
    window_codes = [code_by_residue.get((chain_id, residue), '?') for residue in range(window_start, window_end + 1)]
    target_code = code_by_residue.get((chain_id, target_residue), '?')
    ca = ca_coordinates(array)
    target_bend = float('nan')
    try:
        target_bend = bend_angle(ca[(chain_id, target_residue - 2)], ca[(chain_id, target_residue)], ca[(chain_id, target_residue + 2)])
    except KeyError:
        pass
    deviations = [np.linalg.norm(ca[key] - original_ca[key]) for key in [(chain_id, residue) for residue in range(window_start, window_end + 1)] if key in ca and key in original_ca]
    metadata_clashes = find_numeric_metric(metadata, {'sidechain_clashes', 'num_sidechain_clashes', 'clashes'})
    chainbreaks = count_ca_chainbreaks(array)
    return {
        'target_sse': target_code,
        'window_sse': ''.join(window_codes),
        'window_helix_fraction': float(window_codes.count('a') / len(window_codes)),
        'target_nonhelical': bool(target_code == 'c'),
        'target_bend_angle_deg': target_bend,
        'window_ca_rmsd_from_original': float(np.sqrt(np.mean(np.square(deviations)))) if deviations else float('nan'),
        'window_ca_max_deviation': float(max(deviations)) if deviations else float('nan'),
        'chainbreaks': int(chainbreaks),
        'metadata_sidechain_clashes': metadata_clashes,
        'quality_pass': bool(chainbreaks == 0 and (metadata_clashes is None or metadata_clashes == 0)),
    }

def json_safe(value):
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, (np.integer,)):
        return int(value)
    if isinstance(value, (np.bool_,)):
        return bool(value)
    if isinstance(value, (np.floating,)):
        return None if not np.isfinite(value) else float(value)
    if isinstance(value, float):
        return None if not math.isfinite(value) else value
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, dict):
        return {str(key): json_safe(item) for key, item in value.items()}
    if isinstance(value, (list, tuple)):
        return [json_safe(item) for item in value]
    try:
        json.dumps(value)
        return value
    except TypeError:
        return str(value)

def atom_identity_keys(atom_array):
    array = as_atom_array(atom_array)
    altloc = getattr(array, 'altloc_id', np.full(len(array), ''))
    return [(str(array.chain_id[i]), int(array.res_id[i]), str(array.atom_name[i]), str(altloc[i])) for i in range(len(array))]

def common_topology(atom_arrays):
    arrays = [as_atom_array(array) for array in atom_arrays]
    key_maps = [{key: index for index, key in enumerate(atom_identity_keys(array))} for array in arrays]
    keys = [key for key in atom_identity_keys(arrays[0]) if all(key in mapping for mapping in key_maps[1:])]
    if not keys:
        raise ValueError('The original and generated structures have no common atom topology.')
    indices = [[mapping[key] for key in keys] for mapping in key_maps]
    template = arrays[0][indices[0]].copy()
    coordinates = np.stack([array.coord[index_list] for array, index_list in zip(arrays, indices)], axis=0)
    return template, coordinates, len(keys)

def rigid_superpose(mobile, reference, coordinates):
    mobile_centroid = np.mean(mobile, axis=0)
    reference_centroid = np.mean(reference, axis=0)
    mobile_centered = mobile - mobile_centroid
    reference_centered = reference - reference_centroid
    covariance = mobile_centered.T @ reference_centered
    left, _, right_transpose = np.linalg.svd(covariance)
    rotation = left @ right_transpose
    if np.linalg.det(rotation) < 0.0:
        left[:, -1] *= -1.0
        rotation = left @ right_transpose
    return (coordinates - mobile_centroid) @ rotation + reference_centroid

def choose_alignment_indices(template, coordinates, chain_id, window_start, window_end, mode):
    if mode == 'No alignment':
        return np.array([], dtype=int), 'No alignment'
    if mode not in {'Fixed scaffold outside window', 'All common protein C-alpha'}:
        raise ValueError(f'Unknown ALIGNMENT_MODE: {mode!r}')
    protein_mask = np.asarray(struc.filter_amino_acids(template), dtype=bool)
    atom_names = np.asarray(template.atom_name).astype(str)
    ca_mask = protein_mask & (atom_names == 'CA')
    if mode == 'Fixed scaffold outside window':
        chain_ids = np.asarray(template.chain_id).astype(str)
        residue_ids = np.asarray(template.res_id, dtype=int)
        outside_window = ~((chain_ids == str(chain_id)) & (residue_ids >= int(window_start)) & (residue_ids <= int(window_end)))
        ca_mask &= outside_window
    candidates = np.where(ca_mask)[0]
    if len(candidates):
        finite = np.all(np.isfinite(coordinates[:, candidates, :]), axis=(0, 2))
        candidates = candidates[finite]
    selected_mode = mode
    if len(candidates) < 3 and mode == 'Fixed scaffold outside window':
        fallback = np.where(protein_mask & (atom_names == 'CA'))[0]
        if len(fallback):
            finite = np.all(np.isfinite(coordinates[:, fallback, :]), axis=(0, 2))
            fallback = fallback[finite]
        if len(fallback) >= 3:
            candidates = fallback
            selected_mode = 'All common protein C-alpha (fallback; too few scaffold anchors)'
    if len(candidates) < 3:
        raise ValueError('At least three finite common protein C-alpha atoms are required for alignment.')
    return candidates, selected_mode

def align_multiframe_coordinates(coordinates, template, chain_id, window_start, window_end, mode):
    aligned = np.asarray(coordinates, dtype=float).copy()
    if mode == 'No alignment':
        return aligned, {'mode': 'No alignment', 'atom_count': 0, 'rmsd_angstrom': []}
    indices, selected_mode = choose_alignment_indices(template, aligned, chain_id, window_start, window_end, mode)
    reference = aligned[0, indices].copy()
    rmsds = [0.0]
    for frame_index in range(1, len(aligned)):
        aligned[frame_index] = rigid_superpose(aligned[frame_index, indices], reference, aligned[frame_index])
        anchor_deviation = aligned[frame_index, indices] - reference
        rmsds.append(float(np.sqrt(np.mean(np.sum(anchor_deviation * anchor_deviation, axis=1)))))
    return aligned, {'mode': selected_mode, 'atom_count': int(len(indices)), 'rmsd_angstrom': rmsds}

def write_multiframe_pdb(atom_arrays, path, align_mode, target_chain, window_start, window_end):
    template, coordinates, common_atoms = common_topology(atom_arrays)
    coordinates, alignment_info = align_multiframe_coordinates(coordinates, template, target_chain, window_start, window_end, align_mode)
    pdb = PDBFile()
    pdb.set_structure(struc.from_template(pdb_compatible_array(template), coordinates))
    pdb.write(path)
    return {'common_atoms': int(common_atoms), 'alignment': alignment_info}

def run_rfd3(design_spec, samples, batch_size, seed, step_scale, noise_scale):
    disable_cuequivariance()
    from rfd3.engine import RFD3InferenceConfig, RFD3InferenceEngine
    batches = math.ceil(samples / batch_size)
    config = RFD3InferenceConfig(
        ckpt_path='rfd3',
        diffusion_batch_size=batch_size,
        skip_existing=False,
        prevalidate_inputs=True,
        dump_prediction_metadata_json=True,
        seed=seed,
        specification=design_spec,
        inference_sampler={'step_scale': step_scale, 'noise_scale': noise_scale},
    )
    engine = RFD3InferenceEngine(**config.__dict__)
    outputs_by_example = engine.run(inputs=None, n_batches=batches, out_dir=None)
    outputs = []
    for example_id in sorted(outputs_by_example):
        outputs.extend(outputs_by_example[example_id])
    return outputs[:samples]


In [ ]:
#@title Step 4: Inspect input chains and residue numbering
if INPUT_MODE == 'Sequence':
    preview_sequence = validate_sequence(PROTEIN_SEQUENCE)
    preview_records = [ResidueRecord('A', index, AA1_TO_AA3[letter]) for index, letter in enumerate(preview_sequence, start=1)]
    print(f'Sequence input: {len(preview_sequence)} residues; numbering will be chain A, residues 1–{len(preview_sequence)}.')
else:
    if UPLOADED_STRUCTURE_PATH is None:
        raise RuntimeError('No structure is ready. Run Step 2 and upload a PDB/mmCIF first.')
    preview_structure = load_structure_file(UPLOADED_STRUCTURE_PATH)
    preview_records = residue_records(protein_array(preview_structure))
    if not preview_records:
        raise ValueError('No polymer protein residues were found in that file. Check that it contains a protein chain.')
    print(f'Structure input: {UPLOADED_STRUCTURE_PATH.name}')
    print('Only observed polymer protein residues are listed below; HETATM/non-polymer components remain as context but are not target/window choices.')
    preview_pdb_path = WORKDIR / 'uploaded_structure_preview.pdb'
    write_single_pdb(preview_structure, preview_pdb_path)
    display(HTML('<h3>Uploaded structure preview</h3><p>Inspect the 3D model here, then choose chain and residue IDs from the sequence listing below.</p>'))
    display(HTML(build_molstar_structure_preview_html(preview_pdb_path)))

for chain_id in sorted({record.chain_id for record in preview_records}):
    chain_records = [record for record in preview_records if record.chain_id == chain_id]
    residue_ranges = ', '.join(str(start) if start == end else f'{start}-{end}' for start, end in residue_ids_to_ranges(record.res_id for record in chain_records))
    print(f'\nChain {chain_id or "(blank)"}: {len(chain_records)} observed protein residues; residue IDs {residue_ranges}')
    for offset in range(0, len(chain_records), 10):
        segment = chain_records[offset:offset + 10]
        segment_sequence = ''.join(AA3_TO_AA1.get(record.res_name, 'X') for record in segment)
        print(f'  {segment[0].res_id:>4}-{segment[-1].res_id:<4}  {segment_sequence}')
print('Use these exact chain and residue IDs in the next settings form.')

In [ ]:
#@title Step 5: Choose target, window, mutations, and sampling settings { display-mode: "form" }
TARGET_CHAIN = 'A' #@param {type:"string"}
TARGET_RESIDUE = 36 #@param {type:"integer"}
WINDOW_MODE = 'Manual' #@param ["Manual", "Target helix plus padding"]
WINDOW_START = 30 #@param {type:"integer"}
WINDOW_END = 42 #@param {type:"integer"}
HELIX_PADDING = 1 #@param {type:"integer"}
FALLBACK_HALF_WINDOW = 4 #@param {type:"integer"}
SEQUENCE_MUTATIONS = '' #@param {type:"string"}
# Mutation notation (e.g. N36A) applies only to sequence input. Structure-mode inputs must already be modeled mutants.
N_VARIANTS = 16 #@param {type:"integer"}
# Every candidate is tested against this complete joint glycosylation set.
# ReGlyco receives all attachments in one build, so inter-glycan clashes are
# evaluated together. The loop target/window remains centered on A36.
TARGET_ATTACHMENTS = [
    {'chain': 'A', 'resi': 13, 'glycan': 'G80343LW'},
    {'chain': 'A', 'resi': 36, 'glycan': 'G80343LW'},
    {'chain': 'A', 'resi': 52, 'glycan': 'G88800UV'},
    {'chain': 'A', 'resi': 73, 'glycan': 'G88800UV'},
    {'chain': 'A', 'resi': 83, 'glycan': 'G88800UV'},
]
DIFFUSION_BATCH_SIZE = 4 #@param {type:"integer"}
PARTIAL_T = 12.0 #@param {type:"number"}
STEP_SCALE = 0.95 #@param {type:"number"}
NOISE_SCALE = 1.015 #@param {type:"number"}
RFD3_SEED = 936 #@param {type:"integer"}
EXPOSURE_CONDITION = 'Window' #@param ["Window", "Target residue", "None"]
LOOP_BIAS = 'More loops' #@param ["More loops", "Fewer loops", "None"]
REQUIRE_TARGET_NONHELICAL = True #@param {type:"boolean"}
ALIGNMENT_MODE = 'Fixed scaffold outside window' #@param ["Fixed scaffold outside window", "All common protein C-alpha", "No alignment"]

print('Target:', f'{TARGET_CHAIN}{TARGET_RESIDUE}')
print('Requested window:', f'{TARGET_CHAIN}{WINDOW_START}-{WINDOW_END}' if WINDOW_MODE == 'Manual' else WINDOW_MODE)
print('Accepted glycosylated structures requested:', N_VARIANTS)
print('Joint glycosylation set:', ', '.join(f"{item['chain']}{item['resi']}={item['glycan']}" for item in TARGET_ATTACHMENTS))
if INPUT_MODE == 'PDB/mmCIF structure':
    print('For a structure mutant, upload an already modeled mutant and leave SEQUENCE_MUTATIONS blank.')
else:
    print('Optional sequence mutations use notation such as N36A; the reference residue is validated.')

In [ ]:
#@title Step 6: Prepare input and build the RFD3 specification
input_source_path = None
if INPUT_MODE == 'Sequence':
    wild_type_sequence = validate_sequence(PROTEIN_SEQUENCE)
    sequence, applied_mutations = apply_mutations(wild_type_sequence, SEQUENCE_MUTATIONS)
    print(f'Folding {len(sequence)} residues with RF3...')
    original_atom_array = fold_sequence(sequence, example_id='sequence_input')
    input_description = 'RF3-folded sequence'
else:
    if SEQUENCE_MUTATIONS.strip():
        raise ValueError('SEQUENCE_MUTATIONS applies only to sequence input. For structure mode, upload a structure that already contains the desired mutation; hand-renaming a residue is not a valid side-chain model.')
    structure_path = UPLOADED_STRUCTURE_PATH
    if structure_path is None:
        raise RuntimeError('No structure is ready. Run Step 2 to upload one PDB/mmCIF file, or enter an existing path in STRUCTURE_PATH.')
    input_source_path = Path(structure_path)
    original_atom_array = load_structure_file(structure_path)
    applied_mutations = []
    input_description = f'Structure input: {structure_path.name}'

original_input_pdb = WORKDIR / 'original_input.pdb'
write_single_pdb(original_atom_array, original_input_pdb)

protein = protein_array(original_atom_array)
protein_records = residue_records(protein)
available_chains = sorted({record.chain_id for record in protein_records})
if TARGET_CHAIN not in available_chains:
    raise ValueError(f'Chain {TARGET_CHAIN!r} was not found. Available protein chains: {available_chains}')
residue_index(protein_records, TARGET_CHAIN, TARGET_RESIDUE)

if WINDOW_MODE == 'Target helix plus padding':
    window_start, window_end, window_source = detect_window(original_atom_array, TARGET_CHAIN, TARGET_RESIDUE, HELIX_PADDING, FALLBACK_HALF_WINDOW)
else:
    window_start, window_end, window_source = int(WINDOW_START), int(WINDOW_END), 'manual'
if window_start > window_end:
    raise ValueError('WINDOW_START must be <= WINDOW_END.')
if not window_start <= TARGET_RESIDUE <= window_end:
    raise ValueError('TARGET_RESIDUE must lie inside the selected window.')
chain_residue_ids = {record.res_id for record in protein_records if record.chain_id == TARGET_CHAIN}
for attachment in TARGET_ATTACHMENTS:
    attachment_chain = str(attachment.get('chain', '')).strip()
    attachment_residue = int(attachment.get('resi'))
    if attachment_chain not in available_chains or attachment_residue not in {record.res_id for record in protein_records if record.chain_id == attachment_chain}:
        raise ValueError(f"Joint glycosylation attachment {attachment_chain}{attachment_residue} is not present in the input protein.")
missing_window_residues = [residue for residue in range(window_start, window_end + 1) if residue not in chain_residue_ids]
if missing_window_residues:
    raise ValueError(f'The selected window contains missing/non-contiguous residues: {missing_window_residues}')

# Build the contig from protein residues; RFD3 handles non-protein components as fixed
# partial-diffusion context unless they are explicitly selected otherwise.
all_records = residue_records(protein_array(original_atom_array), protein_only=True)
fixed_selection = make_fixed_selection(all_records, TARGET_CHAIN, window_start, window_end)
if EXPOSURE_CONDITION == 'Window':
    exposed_selection = f'{TARGET_CHAIN}{window_start}-{window_end}'
elif EXPOSURE_CONDITION == 'Target residue':
    exposed_selection = f'{TARGET_CHAIN}{TARGET_RESIDUE}'
else:
    exposed_selection = None

design_spec = {
    'dialect': 2,
    'input': str(original_input_pdb.resolve()),
    'partial_t': float(PARTIAL_T),
    'select_fixed_atoms': fixed_selection,
    'select_unfixed_sequence': False,
    'extra': {
        'method': 'anchor-preserving local conformational ensemble',
        'target_chain': TARGET_CHAIN,
        'target_residue': int(TARGET_RESIDUE),
        'diffuse_window_start': int(window_start),
        'diffuse_window_end': int(window_end),
        'window_source': window_source,
        'mutations': applied_mutations,
    },
}
if exposed_selection is not None:
    design_spec['select_exposed'] = exposed_selection
if LOOP_BIAS == 'More loops':
    design_spec['is_non_loopy'] = False
elif LOOP_BIAS == 'Fewer loops':
    design_spec['is_non_loopy'] = True

SPEC_PATH = WORKDIR / 'design_spec.json'
SPEC_PATH.write_text(json.dumps(json_safe(design_spec), indent=2) + '\n')
ORIGINAL_CA = ca_coordinates(original_atom_array)
print('Input:', input_description)
print('Applied sequence mutations:', applied_mutations or 'none')
print('Window:', f'{TARGET_CHAIN}{window_start}-{window_end}', f'({window_source})')
print('Fixed scaffold:', fixed_selection)
print('RFD3 specification:', SPEC_PATH)

In [ ]:
#@title Step 7: Generate candidates until the accepted glycosylated count is reached
from collections import Counter
from datetime import datetime, timezone

if N_VARIANTS < 1:
    raise ValueError('N_VARIANTS must be at least 1.')
if DIFFUSION_BATCH_SIZE < 1:
    raise ValueError('DIFFUSION_BATCH_SIZE must be at least 1.')
if not TARGET_ATTACHMENTS:
    raise ValueError('TARGET_ATTACHMENTS must contain at least one attachment.')
if any(not str(item.get('glycan', '')).strip() for item in TARGET_ATTACHMENTS):
    raise ValueError('Every TARGET_ATTACHMENTS entry must include a glycan.')

REQUESTED_ACCEPTED = int(N_VARIANTS)
MAX_GENERATED = 20 * REQUESTED_ACCEPTED
ATTACHMENT_SPECS = [f"{item['chain']}:{int(item['resi'])}={item['glycan']}" for item in TARGET_ATTACHMENTS]
ATTACHMENT_SPEC = '; '.join(ATTACHMENT_SPECS)
EXPECTED_SITE_KEYS = {f"{item['resi']}_{item['chain']}" for item in TARGET_ATTACHMENTS}
ATTACHMENT_GLYCANS = [str(item['glycan']) for item in TARGET_ATTACHMENTS]
COLLECTION_ROOT = WORKDIR / 'glycosylation_collection'
RFD3_CANDIDATE_ROOT = COLLECTION_ROOT / 'rfd3_candidates'
RUST_RUN_ROOT = COLLECTION_ROOT / 'reglyco_runs'
ACCEPTED_ROOT = COLLECTION_ROOT / 'accepted'
for directory in (RFD3_CANDIDATE_ROOT, RUST_RUN_ROOT, ACCEPTED_ROOT):
    directory.mkdir(parents=True, exist_ok=True)

accepted_records = []
accepted_arrays = []
rejection_records = []
command_records = []
generated_count = 0
batch_index = 0
collection_started = datetime.now(timezone.utc).isoformat()


def _candidate_seed(candidate_index):
    # A stable derived stream keeps RFD3 and Rust replayable without reusing a
    # seed when a previous candidate was rejected.
    return int(RFD3_SEED) * 1_000_003 + int(candidate_index)


def _save_collection_checkpoint(reason='progress'):
    checkpoint = {
        'schema_version': 1,
        'updated_at': datetime.now(timezone.utc).isoformat(),
        'reason': reason,
        'target': f'{TARGET_CHAIN}{int(TARGET_RESIDUE)}',
        'glycans': ATTACHMENT_GLYCANS,
        'attachments': ATTACHMENT_SPECS,
        'requested_accepted': REQUESTED_ACCEPTED,
        'max_generated': MAX_GENERATED,
        'generated': generated_count,
        'accepted': len(accepted_records),
        'rejections': len(rejection_records),
        'accepted_records': json_safe(accepted_records),
        'rejection_records': json_safe(rejection_records),
        'commands': json_safe(command_records),
    }
    (COLLECTION_ROOT / 'collection_checkpoint.json').write_text(json.dumps(checkpoint, indent=2) + '\n')


def _has_complete_glycosylated_output(candidate_array, output_path):
    if output_path is None or not Path(output_path).is_file() or Path(output_path).stat().st_size == 0:
        return False, None, 'missing output PDB'
    try:
        output_array = load_structure_file(output_path)
    except Exception as exc:
        return False, None, f'output PDB could not be parsed: {exc}'
    # A complete attachment must add the requested glycan atoms to the same
    # protein candidate.  We retain the complete output atom order for the
    # multiframe PDB; no common-topology projection is done here.
    if len(output_array) <= len(candidate_array):
        return False, output_array, 'output contains no additional glycan atoms'
    return True, output_array, None

def _search_site_keys(search):
    keys = set()
    for item in (search or {}).get('sites', []):
        site = item.get('site', {}) if isinstance(item, dict) else {}
        residue = site.get('residue', {}) if isinstance(site, dict) else {}
        if isinstance(residue, dict) and residue.get('chain') is not None and residue.get('number') is not None:
            keys.add(f"{residue['number']}_{residue['chain']}")
    return keys


def _record_rejection(candidate_id, seed, status, reason, candidate_path, metrics, metadata=None, run_dir=None):
    rejection_records.append({
        'candidate_id': candidate_id,
        'seed': int(seed),
        'status': status,
        'reason': str(reason),
        'candidate_pdb': str(candidate_path),
        'metrics': json_safe(metrics),
        'rfd3_metadata': json_safe(metadata or {}),
        'run_dir': str(run_dir) if run_dir is not None else None,
    })


print(f'Collecting {REQUESTED_ACCEPTED} accepted glycosylated structures.')
print(f'Joint attachments: {ATTACHMENT_SPEC}; generation cap: {MAX_GENERATED}.')
try:
    while len(accepted_records) < REQUESTED_ACCEPTED and generated_count < MAX_GENERATED:
        batch_size = min(int(DIFFUSION_BATCH_SIZE), MAX_GENERATED - generated_count)
        batch_seed = int(RFD3_SEED) + batch_index * 1_000_003
        print(f'RFD3 batch {batch_index + 1}: {batch_size} candidate(s), seed {batch_seed}')
        try:
            outputs = run_rfd3(design_spec, batch_size, batch_size, batch_seed, STEP_SCALE, NOISE_SCALE)
        except Exception as exc:
            rejection_records.append({
                'candidate_id': f'batch_{batch_index + 1:04d}',
                'seed': batch_seed,
                'status': 'error',
                'reason': f'RFD3 batch failed: {exc}',
                'candidate_pdb': None,
                'metrics': {},
                'rfd3_metadata': {},
                'run_dir': None,
            })
            _save_collection_checkpoint('rfd3-error')
            print(f'RFD3 batch failed; preserving partial results: {exc}')
            break
        batch_index += 1
        if not outputs:
            rejection_records.append({
                'candidate_id': f'batch_{batch_index:04d}',
                'seed': batch_seed,
                'status': 'error',
                'reason': 'RFD3 returned no structures',
                'candidate_pdb': None,
                'metrics': {},
                'rfd3_metadata': {},
                'run_dir': None,
            })
            _save_collection_checkpoint('rfd3-empty')
            break

        for output in outputs:
            if len(accepted_records) >= REQUESTED_ACCEPTED or generated_count >= MAX_GENERATED:
                break
            generated_count += 1
            candidate_index = generated_count
            candidate_id = f'candidate_{candidate_index:05d}'
            candidate_seed = _candidate_seed(candidate_index)
            candidate_path = RFD3_CANDIDATE_ROOT / f'{candidate_id}.pdb'
            metadata = getattr(output, 'metadata', {}) or {}
            candidate_array = as_atom_array(output.atom_array)
            write_single_pdb(candidate_array, candidate_path)
            metrics = analyze_structure(candidate_array, ORIGINAL_CA, metadata, TARGET_CHAIN, TARGET_RESIDUE, window_start, window_end)
            geometry_pass = bool(metrics['quality_pass'] and (metrics['target_nonhelical'] if REQUIRE_TARGET_NONHELICAL else True))
            if not geometry_pass:
                reason = 'target/window geometry filter failed'
                _record_rejection(candidate_id, candidate_seed, 'geometry_rejected', reason, candidate_path, metrics, metadata)
                _save_collection_checkpoint('geometry-rejected')
                continue

            run_dir = RUST_RUN_ROOT / candidate_id
            run_dir.mkdir(parents=True, exist_ok=True)
            build_args = [
                'build', '--protein', str(candidate_path),
                *sum((["--attach", spec] for spec in ATTACHMENT_SPECS), []),
                '--output', str(run_dir),
                '--no-system', '--allow-clashes',
                '--population', '128', '--generations', '100',
                '--output-format', 'pdb',
            ]
            try:
                result = run_reglyco(build_args, run_dir, seed=candidate_seed, threads=1, check=False)
                command_records.append({
                    'candidate_id': candidate_id,
                    'seed': int(candidate_seed),
                    'result': provenance(result, seed=candidate_seed),
                })
                if not result.ok:
                    _record_rejection(candidate_id, candidate_seed, 'error', f'Rust build failed with exit code {result.returncode}', candidate_path, metrics, metadata, run_dir)
                    _save_collection_checkpoint('rust-error')
                    continue
                search = read_json(run_dir, 'search.json') or {}
                output_path = output_structure(run_dir)
                complete, glyco_array, output_reason = _has_complete_glycosylated_output(candidate_array, output_path)
                status = clash_status(run_dir)
                missing_sites = EXPECTED_SITE_KEYS - _search_site_keys(search)
                if not complete or not is_clash_free(run_dir) or missing_sites:
                    reasons = []
                    if not complete:
                        reasons.append(output_reason or 'incomplete glycosylated output')
                    if not is_clash_free(run_dir):
                        reasons.append(f'search status is {status or "unknown"}')
                    if missing_sites:
                        reasons.append(f'missing joint attachment sites: {", ".join(sorted(missing_sites))}')
                    _record_rejection(candidate_id, candidate_seed, 'glycosylation_rejected', '; '.join(reasons), candidate_path, metrics, {**metadata, 'search': search}, run_dir)
                    _save_collection_checkpoint('glycosylation-rejected')
                    continue
                accepted_path = ACCEPTED_ROOT / f'{candidate_id}.pdb'
                write_single_pdb(glyco_array, accepted_path)
                accepted_arrays.append(glyco_array)
                accepted_records.append({
                    'candidate_id': candidate_id,
                    'seed': int(candidate_seed),
                    'status': 'ClashFree',
                    'glycans': ATTACHMENT_GLYCANS,
                    'attachments': ATTACHMENT_SPECS,
                    'candidate_pdb': str(candidate_path),
                    'glycosylated_pdb': str(accepted_path),
                    'run_dir': str(run_dir),
                    'metrics': json_safe(metrics),
                    'search': json_safe(search),
                    'provenance': json_safe(provenance(result, seed=candidate_seed)),
                })
                _save_collection_checkpoint('accepted')
                print(f'Accepted {candidate_id} ({len(accepted_records)}/{REQUESTED_ACCEPTED})')
            except Exception as exc:
                _record_rejection(candidate_id, candidate_seed, 'error', f'local build exception: {exc}', candidate_path, metrics, metadata, run_dir)
                _save_collection_checkpoint('rust-exception')
        if len(outputs) < batch_size:
            # Do not spin forever if the inference engine returns fewer outputs.
            print(f'RFD3 returned {len(outputs)} structure(s) for a batch of {batch_size}; stopping after this partial batch.')
            break
finally:
    _save_collection_checkpoint('completed-or-interrupted')

VARIANT_ARRAYS = list(accepted_arrays)
rfd3_outputs = []
print(f'Generated {generated_count}; accepted {len(accepted_records)}; rejected/errored {len(rejection_records)}.')
if len(accepted_records) < REQUESTED_ACCEPTED:
    print('The requested accepted count was not reached; partial results are available in the collection checkpoint.')


In [ ]:
#@title Step 8: Analyze accepted glycosylated structures and build the ensemble
reference_metrics = analyze_structure(original_atom_array, ORIGINAL_CA, {}, TARGET_CHAIN, TARGET_RESIDUE, window_start, window_end)
frame_rows = [{
    'frame': 1, 'label': 'Original (unglycosylated)', 'sample_id': 'original', 'status': 'Reference',
    'glycan': None, 'candidate_id': None, 'seed': None,
    **reference_metrics, 'partial_t': 0.0, 'accepted': True,
}]

for index, (record, array) in enumerate(zip(accepted_records, accepted_arrays), start=1):
    metrics = analyze_structure(array, ORIGINAL_CA, record.get('search', {}), TARGET_CHAIN, TARGET_RESIDUE, window_start, window_end)
    metrics.update({
        'frame': index,
        'label': f"Accepted glyco {index:02d}",
        'sample_id': record['candidate_id'],
        'candidate_id': record['candidate_id'],
        'seed': record['seed'],
        'glycan': '; '.join(ATTACHMENT_GLYCANS),
        'glycans': ATTACHMENT_GLYCANS,
        'status': 'ClashFree',
        'partial_t': float(PARTIAL_T),
        'accepted': True,
        'search_status': record.get('status'),
    })
    frame_rows.append(metrics)

FRAME_LABELS = [
    f"{row['label']} | {row.get('glycan') or 'input'} | target SSE={row.get('target_sse', '?')}"
    for row in frame_rows[1:]
]
ACCEPTED_MULTIFRAME_PATH = None
all_frame_info = None
common_atom_count = 0
alignment_info = None
if accepted_arrays:
    ACCEPTED_MULTIFRAME_PATH = WORKDIR / 'accepted_glycosylated_multiframe.pdb'
    all_frame_info = write_multiframe_pdb(accepted_arrays, ACCEPTED_MULTIFRAME_PATH, ALIGNMENT_MODE, TARGET_CHAIN, window_start, window_end)
    common_atom_count = all_frame_info['common_atoms']
    alignment_info = all_frame_info['alignment']

REJECTIONS_PATH = COLLECTION_ROOT / 'rejections.json'
REJECTIONS_PATH.write_text(json.dumps(json_safe(rejection_records), indent=2) + '\n')

RUN_SUMMARY_PATH = WORKDIR / 'run_summary.json'
run_summary = {
    'schema_version': 2,
    'input_description': input_description,
    'mutations': applied_mutations,
    'target': f'{TARGET_CHAIN}{TARGET_RESIDUE}',
    'target_glycans': ATTACHMENT_GLYCANS,
    'attachments': ATTACHMENT_SPECS,
    'requested_accepted': int(REQUESTED_ACCEPTED),
    'max_generated': int(MAX_GENERATED),
    'generated': int(generated_count),
    'geometry_rejected': sum(item.get('status') == 'geometry_rejected' for item in rejection_records),
    'glycosylation_rejected': sum(item.get('status') == 'glycosylation_rejected' for item in rejection_records),
    'errors': sum(item.get('status') == 'error' for item in rejection_records),
    'accepted': len(accepted_records),
    'window': f'{TARGET_CHAIN}{window_start}-{window_end}',
    'alignment': alignment_info,
    'common_atoms_in_accepted_multiframe': int(common_atom_count),
    'design_spec': design_spec,
    'accepted_records': accepted_records,
    'rejections': rejection_records,
    'frames': frame_rows,
    'files': {
        'original_input_pdb': str(original_input_pdb),
        'design_spec_json': str(SPEC_PATH),
        'accepted_frames_pdb': str(ACCEPTED_MULTIFRAME_PATH) if ACCEPTED_MULTIFRAME_PATH else None,
        'rejections_json': str(REJECTIONS_PATH),
        'checkpoint_json': str(COLLECTION_ROOT / 'collection_checkpoint.json'),
    },
}
RUN_SUMMARY_PATH.write_text(json.dumps(json_safe(run_summary), indent=2) + '\n')

RESULTS_DF = pd.DataFrame(frame_rows)
columns = ['frame', 'label', 'status', 'glycan', 'candidate_id', 'seed', 'target_sse', 'window_sse', 'window_helix_fraction', 'target_bend_angle_deg', 'window_ca_rmsd_from_original', 'chainbreaks']
display(RESULTS_DF[[column for column in columns if column in RESULTS_DF]].round(3))
REJECTIONS_DF = pd.DataFrame(rejection_records)
if not REJECTIONS_DF.empty:
    display(REJECTIONS_DF[['candidate_id', 'status', 'reason', 'seed']].head(20))
print(f'Accepted glycosylated structures: {len(accepted_records)} / {REQUESTED_ACCEPTED}')
print(f'Generated candidates: {generated_count} / {MAX_GENERATED}')
print(f'Geometry rejected: {run_summary["geometry_rejected"]}; glycosylation rejected: {run_summary["glycosylation_rejected"]}; errors: {run_summary["errors"]}')
print('Original input remains a separate unglycosylated reference:', original_input_pdb)
if ACCEPTED_MULTIFRAME_PATH:
    print('Accepted glycosylated multiframe:', ACCEPTED_MULTIFRAME_PATH)
else:
    print('No accepted glycosylated structure was produced; inspect rejections and the checkpoint.')


## How to interpret the table

`window_sse` uses Biotite P-SEA codes (`a` = alpha helix, `c` = coil). A variant passes the geometry check when no adjacent numbered Cα pair is farther than 4.5 Å. Every configured attachment is sent to one joint ReGlyco build; the candidate is accepted only when the complete search status is `clash_free` and every configured site is present in the result. When enabled, the acceptance rule also requires the target residue to be classified as coil. The viewer still includes every generated variant so you can inspect near-misses and adjust `PARTIAL_T`, loop guidance, or the selected window.

For a mutation series, use the same window, flanking scaffold, sample count, parameter grid, and seed policy for each sequence/structure. Run the wild type and mutant separately, then compare accepted-state frequencies and geometry; those frequencies are protocol-dependent and are not thermodynamic populations.

In [ ]:
#@title Step 9: Inspect aligned models in Mol*
def build_molstar_multiframe_html(pdb_path, frame_labels, height=700):
    pdb_text = Path(pdb_path).read_text()
    payload = {
        'pdb_base64': base64.b64encode(pdb_text.encode('utf-8')).decode('ascii'),
        'labels': list(frame_labels),
    }
    token = uuid.uuid4().hex
    viewer_id = f'local-ensemble-viewer-{token}'
    select_id = f'local-ensemble-select-{token}'
    all_button_id = f'local-ensemble-all-{token}'
    loading_id = f'local-ensemble-loading-{token}'
    html = '''
<style>
.local-ensemble-wrap { font-family: system-ui, -apple-system, Segoe UI, Roboto, Arial; }
.local-ensemble-toolbar { display:flex; align-items:center; gap:10px; margin:8px 0; flex-wrap:wrap; }
.local-ensemble-toolbar select, .local-ensemble-toolbar button { padding:7px 10px; border:1px solid #cbd5e1; border-radius:8px; background:#fff; cursor:pointer; }
.local-ensemble-note { color:#475569; font-size:12px; margin:4px 0 8px; }
#__VIEWER_ID__ { width:100%; height:__HEIGHT__px; position:relative; border:1px solid #dbe3ec; border-radius:12px; overflow:hidden; }
.local-ensemble-loading { position:absolute; inset:0; z-index:5; display:flex; align-items:center; justify-content:center; background:#fff; color:#64748b; letter-spacing:.08em; font-size:12px; }
</style>
<link rel='stylesheet' href='https://cdn.jsdelivr.net/npm/molstar@3/build/viewer/molstar.css'>
<script src='https://cdn.jsdelivr.net/npm/molstar@3/build/viewer/molstar.js'></script>
<div class='local-ensemble-wrap'>
  <div class='local-ensemble-toolbar'>
    <label for='__SELECT_ID__'>Frame:</label>
    <select id='__SELECT_ID__'></select>
    <button id='__ALL_BUTTON_ID__'>Show all models</button>
  </div>
  <div class='local-ensemble-note'>The accepted glycosylated multiframe is aligned to the original protein scaffold using stable C-alpha anchors outside the selected window. The original unglycosylated input is shown separately so glycan atoms remain in every accepted model.</div>
  <div id='__VIEWER_ID__'><div id='__LOADING_ID__' class='local-ensemble-loading'>LOADING MOL*</div></div>
</div>
<script>
(() => {
  const payload = __PAYLOAD__;
  const viewerId = '__VIEWER_ID__';
  const selectId = '__SELECT_ID__';
  const allButtonId = '__ALL_BUTTON_ID__';
  const loadingId = '__LOADING_ID__';
  const text = new TextDecoder().decode(Uint8Array.from(atob(payload.pdb_base64), c => c.charCodeAt(0)));
  let viewer = null;
  let structures = [];
  const byId = id => document.getElementById(id);
  function fillSelector() {
    const select = byId(selectId);
    if (!select) return;
    payload.labels.forEach((label, index) => {
      const option = document.createElement('option');
      option.value = String(index);
      option.textContent = label;
      select.appendChild(option);
    });
  }
  function setHidden(index) {
    if (!viewer || !structures.length) return;
    structures.forEach((structure, current) => viewer.plugin.state.data.updateCellState(structure.ref, { isHidden: current !== index }));
    if (viewer.plugin.managers.camera.reset) viewer.plugin.managers.camera.reset();
  }
  function showAll() {
    if (!viewer || !structures.length) return;
    structures.forEach(structure => viewer.plugin.state.data.updateCellState(structure.ref, { isHidden: false }));
    if (viewer.plugin.managers.camera.reset) viewer.plugin.managers.camera.reset();
  }
  async function init() {
    try {
      viewer = await molstar.Viewer.create(viewerId, { layoutIsExpanded:false, layoutShowControls:true, layoutShowRemoteState:false, layoutShowSequence:true, layoutShowLog:false });
      const data = await viewer.plugin.builders.data.rawData({ data: text, label: 'Accepted glycosylated structures' });
      const trajectory = await viewer.plugin.builders.structure.parseTrajectory(data, 'pdb');
      const result = await viewer.plugin.builders.structure.hierarchy.applyPreset(trajectory, 'all-models', { useDefaultIfSingleModel:true, showUnitcell:false, representationPreset:'auto' });
      structures = result && result.structures ? result.structures : [];
      fillSelector();
      const select = byId(selectId);
      if (select) select.addEventListener('change', () => setHidden(Number(select.value)));
      const allButton = byId(allButtonId);
      if (allButton) allButton.addEventListener('click', showAll);
      showAll();
      const loading = byId(loadingId);
      if (loading) loading.style.display = 'none';
    } catch (error) {
      console.error('Mol* multiframe load failed:', error);
      const loading = byId(loadingId);
      if (loading) loading.textContent = 'Mol* failed to load; the PDB is still available below.';
    }
  }
  function waitForMolstar(tries = 0) {
    if (typeof molstar !== 'undefined' && molstar.Viewer) { init(); return; }
    if (tries < 60) { setTimeout(() => waitForMolstar(tries + 1), 200); return; }
    const loading = byId(loadingId);
    if (loading) loading.textContent = 'Mol* JavaScript did not load; use the PDB link below.';
  }
  waitForMolstar();
})();
</script>
'''
    return (html.replace('__PAYLOAD__', json.dumps(payload))
                .replace('__VIEWER_ID__', viewer_id)
                .replace('__SELECT_ID__', select_id)
                .replace('__ALL_BUTTON_ID__', all_button_id)
                .replace('__LOADING_ID__', loading_id)
                .replace('__HEIGHT__', str(int(height))))

if ACCEPTED_MULTIFRAME_PATH is not None:
    display(HTML(build_molstar_multiframe_html(ACCEPTED_MULTIFRAME_PATH, FRAME_LABELS)))
else:
    print('No accepted glycosylated frames are available for Mol*. Review the rejection table and collection checkpoint.')
print('Original unglycosylated input:')
display(HTML(build_molstar_structure_preview_html(original_input_pdb, height=520)))

In [ ]:
#@title Step 10: Download final files
import zipfile
from IPython.display import clear_output
from google.colab import files, output as colab_output
import ipywidgets as widgets

colab_output.enable_custom_widget_manager()

RESULTS_ZIP_PATH = WORKDIR / 'local_conformational_ensemble_results.zip'
bundle_entries = [
    ('inputs/original_input.pdb', original_input_pdb),
    ('metadata/design_spec.json', SPEC_PATH),
    ('metadata/run_summary.json', RUN_SUMMARY_PATH),
    ('metadata/rejections.json', REJECTIONS_PATH),
    ('metadata/collection_checkpoint.json', COLLECTION_ROOT / 'collection_checkpoint.json'),
]
if input_source_path is not None and Path(input_source_path).is_file():
    bundle_entries.append((f'inputs/uploaded_original/{Path(input_source_path).name}', Path(input_source_path)))
if ACCEPTED_MULTIFRAME_PATH is not None:
    bundle_entries.append(('ensembles/accepted_glycosylated_multiframe.pdb', ACCEPTED_MULTIFRAME_PATH))
for record in accepted_records:
    candidate_id = record['candidate_id']
    accepted_path = Path(record['glycosylated_pdb'])
    candidate_path = Path(record['candidate_pdb'])
    run_dir = Path(record['run_dir'])
    bundle_entries.append((f'accepted/{accepted_path.name}', accepted_path))
    bundle_entries.append((f'candidates/{candidate_path.name}', candidate_path))
    for filename in ('search.json', 'report.json', 'glycoprotein.pdb', 'reglyco.stdout.log', 'reglyco.stderr.log'):
        path = run_dir / filename
        if path.is_file():
            bundle_entries.append((f'reglyco_runs/{candidate_id}/{filename}', path))
for record in rejection_records:
    candidate_path = record.get('candidate_pdb')
    if candidate_path and Path(candidate_path).is_file():
        bundle_entries.append((f'rejected_candidates/{Path(candidate_path).name}', Path(candidate_path)))
with zipfile.ZipFile(RESULTS_ZIP_PATH, 'w', compression=zipfile.ZIP_DEFLATED) as result_zip:
    seen = set()
    for archive_name, file_path in bundle_entries:
        file_path = Path(file_path)
        if archive_name in seen or not file_path.is_file():
            continue
        result_zip.write(file_path, arcname=archive_name)
        seen.add(archive_name)

download_items = [
    ('Download all results (.zip)', RESULTS_ZIP_PATH),
    ('Download processed original input (.pdb)', original_input_pdb),
    ('Download run summary (.json)', RUN_SUMMARY_PATH),
    ('Download rejection records (.json)', REJECTIONS_PATH),
    ('Download RFD3 specification (.json)', SPEC_PATH),
]
if ACCEPTED_MULTIFRAME_PATH is not None:
    download_items.insert(1, ('Download accepted glycosylated ensemble (.pdb)', ACCEPTED_MULTIFRAME_PATH))

download_status = widgets.Output()
download_buttons = []
for item_index, (label, file_path) in enumerate(download_items):
    button = widgets.Button(description=label, button_style='primary' if item_index == 0 else '', layout=widgets.Layout(width='390px'))
    def make_download_handler(path):
        def handle_download(_button):
            with download_status:
                clear_output(wait=True)
                if not Path(path).is_file():
                    print(f'File not found: {path}')
                    return
                print(f'Starting download: {Path(path).name}')
            try:
                files.download(str(path))
            except Exception as exc:
                with download_status:
                    clear_output(wait=True)
                    print(f'Download failed: {exc}')
        return handle_download
    button.on_click(make_download_handler(file_path))
    download_buttons.append(button)

display(widgets.HTML('<h3>Download your results</h3><p>The ZIP contains the original input, complete accepted glycosylated structures, candidate/rejection records, Rust reports, commands, and the collection checkpoint. The original unglycosylated structure is kept separate from the accepted multiframe ensemble.</p>'))
display(widgets.VBox(download_buttons))
display(download_status)
